In [2]:
# ============================================================================
# CELL 1: KHỞI TẠO SPARK SESSION TỐI ƯU
# ============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder \
    .appName("EV_Charging_Preprocessing_Optimized") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.sql.files.maxPartitionBytes", "128m") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

# Điều chỉnh log level
spark.sparkContext.setLogLevel("WARN")

print("="*80)
print("PREPROCESSING PIPELINE - OPTIMIZED FOR SCALE")
print("="*80)

26/04/13 02:12:41 WARN Utils: Your hostname, ax2 resolves to a loopback address: 127.0.1.1; using 192.168.40.130 instead (on interface ens33)
26/04/13 02:12:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 02:12:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PREPROCESSING PIPELINE - OPTIMIZED FOR SCALE


In [5]:
# ============================================================================
# CELL 2: ĐỌC DỮ LIỆU VỚI PARTITION HỢP LÝ
# ============================================================================
import builtins

print("\n📂 STEP 1: LOADING DATA WITH OPTIMAL PARTITIONS")

# Đọc file JSON
df = spark.read \
    .option("recursiveFileLookup", "true") \
    .json("hdfs://localhost:9000/ev-project/data/bronze/ev_sessions/caltech/*/*/*")

# Coalesce để có số partition hợp lý
initial_partitions = df.rdd.getNumPartitions()
print(f"✓ Initial partitions: {initial_partitions}")

# Điều chỉnh số partition (giữa 10 và 50)
target_partitions = builtins.max(10, builtins.min(initial_partitions, 50))
df = df.coalesce(target_partitions)
print(f"✓ Adjusted partitions: {df.rdd.getNumPartitions()}")

# Đếm số rows (có thể mất thời gian với data lớn)
rows_total = df.count()
print(f"✓ Loaded {rows_total:,} rows")


📂 STEP 1: LOADING DATA WITH OPTIMAL PARTITIONS


✓ Initial partitions: 181
✓ Adjusted partitions: 50


[Stage 8:======================================================>  (48 + 2) / 50]

✓ Loaded 31,424 rows


In [6]:
# ============================================================================
# CELL 3: DATA CLEANING - PERSIST STRATEGICALLY
# ============================================================================
print("\n" + "="*80)
print("STEP 2: DATA CLEANING & TYPE CONVERSION")
print("="*80)

# 1. Drop ID columns (thực hiện sớm để giảm memory)
id_cols = ["_id", "sessionID", "stationID", "spaceID", "siteID", 
           "clusterID", "userID", "_batch_id", "_ingest_time"]
existing_id_cols = [c for c in id_cols if c in df.columns]
df_clean = df.drop(*existing_id_cols)
print(f"✓ Dropped {len(existing_id_cols)} ID columns")

# 2. Convert timestamps - dùng UDFs hiệu quả hơn
def clean_timestamp(ts_col):
    """Xử lý timestamp với regex hiệu quả"""
    return to_timestamp(
        regexp_replace(col(ts_col), "-07:00|-[0-9]{2}:[0-9]{2}$", ""),
        "yyyy-MM-dd HH:mm:ss"
    )

for tc in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    if tc in df_clean.columns:
        df_clean = df_clean.withColumn(tc, clean_timestamp(tc))

print(f"✓ Converted timestamp columns")

# Persist ở mức độ vừa phải (MEMORY_ONLY cho data vừa, DISK_ONLY cho data lớn)
df_clean.persist(StorageLevel.MEMORY_AND_DISK)
print(f"✓ Data persisted with {df_clean.rdd.getNumPartitions()} partitions")


STEP 2: DATA CLEANING & TYPE CONVERSION
✓ Dropped 9 ID columns
✓ Converted timestamp columns


[Stage 11:==================================================>     (45 + 4) / 50]

✓ Data persisted with 50 partitions


[Stage 11:=====================================================>  (48 + 2) / 50]

In [7]:
# ============================================================================
# CELL 4: HANDLE MISSING VALUES - OPTIMIZED
# ============================================================================
print("\n" + "="*80)
print("STEP 3: HANDLE MISSING VALUES (OPTIMIZED)")
print("="*80)

rows_before = df_clean.count()

# Sử dụng cache count để tránh tính toán lại
null_counts = df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in ["doneChargingTime", "kWhDelivered"]
]).collect()[0]

print(f"✓ NULLs before handling:")
print(f"  - doneChargingTime: {null_counts['doneChargingTime']:,} rows")
print(f"  - kWhDelivered: {null_counts['kWhDelivered']:,} rows")

# Fill NULLs
df_clean = df_clean.withColumn(
    "doneChargingTime",
    coalesce(col("doneChargingTime"), col("disconnectTime"))
)

# Drop remaining NULLs trong 1 lần (hiệu quả hơn)
critical_cols = ["connectionTime", "disconnectTime", "doneChargingTime", "kWhDelivered"]
df_clean = df_clean.dropna(subset=critical_cols)

rows_after = df_clean.count()
print(f"\n✓ Rows after handling: {rows_after:,}")
print(f"✓ Dropped: {rows_before - rows_after:,} rows ({(rows_before-rows_after)/rows_before*100:.2f}%)")


STEP 3: HANDLE MISSING VALUES (OPTIMIZED)


✓ NULLs before handling:
  - doneChargingTime: 2,055 rows
  - kWhDelivered: 0 rows

✓ Rows after handling: 31,424
✓ Dropped: 0 rows (0.00%)


In [8]:
# ============================================================================
# CELL 5: FEATURE ENGINEERING - 17 FEATURES EXACTLY AS PAPER
# ============================================================================
print("\n" + "="*80)
print("STEP 4: FEATURE ENGINEERING (17 FEATURES)")
print("="*80)

# Cache connectionTime để dùng nhiều lần
conn_time = col("connectionTime")
disconn_time = col("disconnectTime")
done_time = col("doneChargingTime")

# Duration features (hours)
df_clean = df_clean.withColumn("duration", 
    (unix_timestamp(disconn_time) - unix_timestamp(conn_time)) / 3600)
df_clean = df_clean.withColumn("charging_duration",
    (unix_timestamp(done_time) - unix_timestamp(conn_time)) / 3600)

# Remove invalid durations (filter early)
df_clean = df_clean.filter((col("duration") > 0) & (col("charging_duration") > 0))

# Time features
df_clean = df_clean \
    .withColumn("hour", hour(conn_time)) \
    .withColumn("day_of_week", dayofweek(conn_time) - 1) \
    .withColumn("month", month(conn_time)) \
    .withColumn("day_of_year", dayofyear(conn_time)) \
    .withColumn("week_of_year", weekofyear(conn_time))

# Binary features
df_clean = df_clean \
    .withColumn("is_weekend", when(col("day_of_week").isin([5,6]), 1).otherwise(0)) \
    .withColumn("is_holiday", when(col("month").isin([12,1]), 1).otherwise(0))

# Season (0=Winter,1=Spring,2=Summer,3=Fall)
df_clean = df_clean.withColumn("season",
    when(col("month").isin([12,1,2]), 0)
    .when(col("month").isin([3,4,5]), 1)
    .when(col("month").isin([6,7,8]), 2)
    .otherwise(3))

# Cyclical features (using pi())
df_clean = df_clean \
    .withColumn("hour_sin", sin(2 * pi() * col("hour") / 24)) \
    .withColumn("hour_cos", cos(2 * pi() * col("hour") / 24))

# Log-transformed feature (with epsilon)
epsilon = 1e-6
df_clean = df_clean.withColumn("charging_duration_log", 
    log(col("charging_duration") + epsilon))

print("✓ All 11 base features created")
print(f"  - Intermediate row count: {df_clean.count():,}")
print(f"  - Columns so far: {len(df_clean.columns)}")


STEP 4: FEATURE ENGINEERING (17 FEATURES)
✓ All 11 base features created
  - Intermediate row count: 31,396
  - Columns so far: 19


In [9]:
# ============================================================================
# CELL 6: OUTLIER HANDLING - IQR WITH APPROXIMATE QUANTILES
# ============================================================================
print("\n" + "="*80)
print("STEP 5: OUTLIER HANDLING (IQR WITH APPROXIMATE QUANTILES)")
print("="*80)

def cap_outliers_optimized(df, col_name):
    """Cap outliers using approxQuantile (nhanh hơn cho data lớn)"""
    if col_name not in df.columns:
        return df
    
    # Dùng approxQuantile thay vì tính toán chính xác (nhanh hơn nhiều)
    quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
    if len(quantiles) < 2 or quantiles[0] == quantiles[1]:
        return df
    
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    # Count outliers (dùng filter thay vì count trên toàn bộ)
    outliers = df.filter((col(col_name) < lower) | (col(col_name) > upper)).count()
    print(f"  {col_name}: {outliers} outliers ({outliers/df.count()*100:.2f}%)")
    
    # Cap values using when/otherwise
    return df.withColumn(
        col_name,
        when(col(col_name) < lower, lower)
        .when(col(col_name) > upper, upper)
        .otherwise(col(col_name))
    )

# Apply to all three columns
for col_name in ["duration", "charging_duration", "kWhDelivered"]:
    df_clean = cap_outliers_optimized(df_clean, col_name)

print("✓ Outlier handling complete")


STEP 5: OUTLIER HANDLING (IQR WITH APPROXIMATE QUANTILES)
  duration: 438 outliers (1.40%)


  charging_duration: 2142 outliers (6.82%)


  kWhDelivered: 1550 outliers (4.94%)
✓ Outlier handling complete


In [10]:
# ============================================================================
# CELL 7: LAG & ROLLING FEATURES - WITH PROPER PARTITIONING
# ============================================================================
print("\n" + "="*80)
print("STEP 6: LAG & ROLLING FEATURES (CRITICAL - MUST BE PARTITIONED)")
print("="*80)

# ⚠️ QUAN TRỌNG: Phải có partitionBy để tránh shuffle toàn bộ dữ liệu
# Nếu không có userID, dùng combination các trường để tạo key
if "userID" not in df_clean.columns:
    # Tạo partition key từ timestamp và station (giả định)
    df_clean = df_clean.withColumn("partition_key", 
        hash(col("connectionTime")) % 50)  # 50 partitions
    partition_cols = ["partition_key"]
else:
    partition_cols = ["userID"]

# Window spec với partition (TRÁNH WARNING VÀ TĂNG PERFORMANCE)
window_spec = Window.partitionBy(*partition_cols).orderBy("connectionTime")

print(f"✓ Window partitioned by: {partition_cols}")

# Create lag features (1, 2, 3)
lag_features = []
for i in [1, 2, 3]:
    lag_col = f"lag_{i}_log"
    df_clean = df_clean.withColumn(lag_col, 
        lag("charging_duration_log", i).over(window_spec))
    lag_features.append(lag_col)
    print(f"✓ Created {lag_col}")

# Rolling mean (3)
df_clean = df_clean.withColumn("rolling_mean_3_log",
    (col("lag_1_log") + col("lag_2_log") + col("lag_3_log")) / 3)

# Rolling mean (5) - dùng rowsBetween để giới hạn window
window_5 = Window.partitionBy(*partition_cols).orderBy("connectionTime") \
    .rowsBetween(-5, -1)

df_clean = df_clean.withColumn("rolling_mean_5_log",
    avg("charging_duration_log").over(window_5))

print("✓ Created rolling_mean_3_log and rolling_mean_5_log")

# Drop rows with NULLs (xảy ra ở đầu mỗi partition)
rows_before_lag = df_clean.count()
df_clean = df_clean.dropna(subset=lag_features + ["rolling_mean_3_log", "rolling_mean_5_log"])
rows_after_lag = df_clean.count()
print(f"✓ Dropped {rows_before_lag - rows_after_lag:,} rows due to lag features")

# Drop temporary column
if "partition_key" in df_clean.columns:
    df_clean = df_clean.drop("partition_key")


STEP 6: LAG & ROLLING FEATURES (CRITICAL - MUST BE PARTITIONED)
✓ Window partitioned by: ['partition_key']
✓ Created lag_1_log
✓ Created lag_2_log
✓ Created lag_3_log
✓ Created rolling_mean_3_log and rolling_mean_5_log


[Stage 53:>                                                         (0 + 1) / 1]

✓ Dropped 297 rows due to lag features


In [11]:
# ============================================================================
# CELL 8: FINAL DATASET - EXACTLY 17 FEATURES + TARGET
# ============================================================================
print("\n" + "="*80)
print("STEP 7: FINAL DATASET PREPARATION (17 FEATURES)")
print("="*80)

# Exactly 17 features as per paper
final_features = [
    "hour", "day_of_week", "month", "season",
    "duration", "charging_duration", "charging_duration_log",
    "hour_sin", "hour_cos", "day_of_year", "week_of_year",
    "is_holiday",
    "lag_1_log", "lag_2_log", "lag_3_log",
    "rolling_mean_3_log", "rolling_mean_5_log"
]

# Verify we have all features
missing_features = [f for f in final_features if f not in df_clean.columns]
if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

print(f"✓ All 17 features verified:")
for i, f in enumerate(final_features, 1):
    print(f"  {i:2d}. {f}")

# Select final dataset
df_final = df_clean.select(final_features + ["kWhDelivered", "connectionTime"])

# Cache final dataset for multiple uses
df_final.persist(StorageLevel.MEMORY_AND_DISK)

final_count = df_final.count()
print(f"\n✓ Final dataset: {final_count:,} rows, {len(df_final.columns)} columns")


STEP 7: FINAL DATASET PREPARATION (17 FEATURES)
✓ All 17 features verified:
   1. hour
   2. day_of_week
   3. month
   4. season
   5. duration
   6. charging_duration
   7. charging_duration_log
   8. hour_sin
   9. hour_cos
  10. day_of_year
  11. week_of_year
  12. is_holiday
  13. lag_1_log
  14. lag_2_log
  15. lag_3_log
  16. rolling_mean_3_log
  17. rolling_mean_5_log


[Stage 61:==============================================>       (173 + 4) / 200]


✓ Final dataset: 31,099 rows, 19 columns


In [12]:
# ============================================================================
# CELL 9: OPTIONAL - VECTOR ASSEMBLER FOR ML (IF NEEDED)
# ============================================================================
print("\n" + "="*80)
print("STEP 8: VECTOR ASSEMBLER (OPTIONAL - FOR ML PIPELINE)")
print("="*80)

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# Feature assembler (chỉ dùng nếu chuẩn bị train model)
assembler = VectorAssembler(
    inputCols=final_features,
    outputCol="features",
    handleInvalid="keep"  # Không có invalid trong dataset đã clean
)

# Optional: Scaling (nếu paper yêu cầu)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False  # sparse data thường không mean-center
)

# Apply assembler
df_assembled = assembler.transform(df_final)

print(f"✓ VectorAssembler created with {len(final_features)} features")
print(f"✓ Feature vector length: {len(final_features)}")

# Show sample
df_assembled.select("features", "kWhDelivered").show(5, truncate=False)


STEP 8: VECTOR ASSEMBLER (OPTIONAL - FOR ML PIPELINE)
✓ VectorAssembler created with 17 features
✓ Feature vector length: 17


26/04/13 02:23:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|features                                                                                                                                                                                                                        |kWhDelivered|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+
|[18.0,2.0,5.0,1.0,11.224444444444444,7.8533333333333335,2.3571522839954655,-1.0,-1.8369701987210297E-16,121.0,18.0,0.0,-2.0148955205703896,0.0736127463750724,1.200132406094378,-0.247050122700313,-0.247050122700313]          |13.95       |
|[8.0,3.0,5.0,1.0,8.970277777777778,1.71

In [14]:
# ============================================================================
# CELL 10: SAVE TO SILVER LAYER (FIXED - NO "m" IN NUMBERS)
# ============================================================================
print("\n" + "="*80)
print("STEP 9: SAVING TO SILVER LAYER (PARTITIONED BY YEAR/MONTH)")
print("="*80)

silver_output_path = "hdfs://localhost:9000/ev-project/data/silver/ev_sessions_preprocessed"

# Add partition columns
df_final = df_final \
    .withColumn("year", year("connectionTime")) \
    .withColumn("month", month("connectionTime"))

# Save - VERSION ĐƠN GIẢN (Khuyến nghị)
df_final.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(silver_output_path)

print(f"✓ Saved to: {silver_output_path}")

# Verify saved data
saved_df = spark.read.parquet(silver_output_path)
print(f"✓ Verification: {saved_df.count():,} rows")


STEP 9: SAVING TO SILVER LAYER (PARTITIONED BY YEAR/MONTH)


✓ Saved to: hdfs://localhost:9000/ev-project/data/silver/ev_sessions_preprocessed


[Stage 74:====================================================> (188 + 4) / 192]

✓ Verification: 31,099 rows


In [17]:
# ============================================================================
# CELL 11: FINAL VALIDATION & STATISTICS (COMPLETE FIXED VERSION)
# ============================================================================
import builtins
from pyspark.sql.functions import col, min, max, mean, stddev, percentile_approx, corr

print("\n" + "="*80)
print("FINAL VALIDATION")
print("="*80)

# Check NULLs
print("\n📊 NULL value check:")
all_columns = final_features + ["kWhDelivered"]
null_results = []

for col_name in all_columns:
    null_count = df_final.filter(col(col_name).isNull()).count()
    is_clean = null_count == 0
    null_results.append((col_name, null_count, is_clean))
    status = "✓" if is_clean else "⚠"
    print(f"  {status} {col_name}: {'clean' if is_clean else f'{null_count} NULLs'}")

if all(is_clean for _, _, is_clean in null_results):
    print("\n✅ NO NULLS FOUND - Dataset is clean!")

# Target variable statistics
print("\n📊 Target variable (kWhDelivered) statistics:")
df_final.select(
    min("kWhDelivered").alias("min"),
    max("kWhDelivered").alias("max"),
    mean("kWhDelivered").alias("mean"),
    stddev("kWhDelivered").alias("stddev"),
    percentile_approx("kWhDelivered", 0.5).alias("median"),
    percentile_approx("kWhDelivered", 0.25).alias("q1"),
    percentile_approx("kWhDelivered", 0.75).alias("q3")
).show()

# Feature correlations with target
print("\n📊 Feature correlations with kWhDelivered:")
correlations = []

for feature in final_features:
    try:
        corr_value = df_final.select(corr(feature, "kWhDelivered")).collect()[0][0]
        if corr_value is not None and not pd.isna(corr_value):  # pd import if needed
            correlations.append((feature, builtins.abs(corr_value), corr_value))
    except Exception as e:
        print(f"  Could not compute correlation for {feature}: {e}")

# Sort and display
if correlations:
    correlations.sort(key=lambda x: x[1], reverse=True)
    print("\nTop features by absolute correlation:")
    for feature, abs_corr, corr_val in correlations[:5]:
        sign = "+" if corr_val > 0 else "-"
        print(f"  {feature:25s}: {sign}{abs_corr:.4f}")
    
    # Also show bottom features
    print("\nBottom features (lowest correlation):")
    for feature, abs_corr, corr_val in correlations[-3:]:
        sign = "+" if corr_val > 0 else "-"
        print(f"  {feature:25s}: {sign}{abs_corr:.4f}")
else:
    print("  No correlations computed")

print("\n" + "="*80)
print("✅ VALIDATION COMPLETE!")
print("="*80)


FINAL VALIDATION

📊 NULL value check:


  ✓ hour: clean


  ✓ day_of_week: clean


  ✓ month: clean
  ✓ season: clean


  ✓ duration: clean


...

  Could not compute correlation for lag_2_log: name 'pd' is not defined


  Could not compute correlation for lag_3_log: name 'pd' is not defined


  Could not compute correlation for rolling_mean_3_log: name 'pd' is not defined


[Stage 435:===========================================>         (163 + 4) / 200]

  Could not compute correlation for rolling_mean_5_log: name 'pd' is not defined
  No correlations computed

✅ VALIDATION COMPLETE!


In [19]:
# ============================================================================
# CELL 12: PERFORMANCE METRICS & CLEANUP (SIMPLIFIED)
# ============================================================================
print("\n" + "="*80)
print("PERFORMANCE METRICS")
print("="*80)

# Basic Spark info
print(f"📈 Spark Application ID: {spark.sparkContext.applicationId}")
print(f"📈 Spark UI: http://localhost:4040")
print(f"📈 Master: {spark.sparkContext.master}")
print(f"📈 Default parallelism: {spark.sparkContext.defaultParallelism}")

# Dataset info
print(f"📈 Final dataset partitions: {df_final.rdd.getNumPartitions()}")
print(f"📈 Final dataset rows: {df_final.count():,}")

# Memory info (optional)
try:
    from pyspark import SparkContext
    # Lấy memory status dưới dạng dict
    memory_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
    # Convert to dict for better display
    executors = {}
    for entry in memory_status.entrySet():
        key = entry.getKey()
        value = entry.getValue()
        executors[str(key)] = {
            "max_memory": value._1(),  # max memory
            "used_memory": value._2()   # used memory
        }
    print(f"📈 Number of executors: {len(executors)}")
except Exception as e:
    print(f"📈 (Detailed memory info not available: {type(e).__name__})")

# Cleanup
print("\n🧹 Cleaning up cached data...")
df_final.unpersist()
if 'df_clean' in locals():
    df_clean.unpersist()
print("✓ Cache cleared")

print("\n" + "="*80)
print("✅ PREPROCESSING COMPLETE!")
print("="*80)


PERFORMANCE METRICS
📈 Spark Application ID: local-1776046366036
📈 Spark UI: http://localhost:4040
📈 Master: local[*]
📈 Default parallelism: 200
📈 Final dataset partitions: 200


[Stage 440:===========================================>         (166 + 4) / 200]

📈 Final dataset rows: 31,099
📈 (Detailed memory info not available: Py4JError)

🧹 Cleaning up cached data...
✓ Cache cleared

✅ PREPROCESSING COMPLETE!
